# Unit 2, Lecture 5: Context-aware agents and testing

The last lecture of Unit 2, and two threads that belong together.

**Memory** is what lets an agent hold a conversation: "route this to billing"
then "make it urgent" only works if the second turn remembers the first.

**Testing** is hard *because* of that history, and because the model words its
answers differently every run. You cannot assert exact output. So you test the
things that are stable: the invariants that must always hold.

All offline, no model.

## Setup

In [ ]:
from cse476.context import (
    Session, ContextAgent, needs_context,
    check_invariants, all_passed,
    queue_is_valid, field_present, field_in,
)

agent = ContextAgent()
print("fresh agent, empty session:", agent.session.turns)

## 1. An agent that carries context across turns

Watch the second turn understand "it" by looking back at the first.

In [ ]:
print("turn 1:", agent.handle("route ticket 12 to billing"))
print("turn 2:", agent.handle("make it urgent"))
print()
print("what the session pinned:")
print("  queue:   ", agent.session.recall("queue"))
print("  priority:", agent.session.recall("priority"))

The word "it" carried no meaning on its own. The session did: it remembered
that the current ticket was routed to billing, so "make it urgent" resolved to
"make the billing ticket urgent".

## 2. The honest failure

If the very first thing a user says is "make it urgent", there is no earlier
ticket. A good agent does **not** invent one.

In [ ]:
fresh = ContextAgent()
print(fresh.handle("make it urgent"))

## 3. Memory is bounded, and pinned facts survive trimming

A session keeps two kinds of memory: a **bounded transcript** (oldest turns drop)
and **pinned facts** kept separate. This is the pinning idea from Unit 1 Lecture
3, now given a home.

In [ ]:
# a tiny session that only keeps the last 2 turns
sess = Session(max_turns=2)
sess.remember("queue", "abuse")          # pin a fact

for i in range(5):
    sess.add("user", f"message {i}")

print("transcript (trimmed to 2):")
print(sess.transcript())
print()
print("but the pinned fact survived:", sess.recall("queue"))

The transcript lost everything but the last two turns. The pinned `queue`
survived, because it lives separately. **If you only kept the transcript, you
would eventually trim away the one fact you needed.**

## 4. Why the obvious test is broken

Here is the test everyone writes first, and why it fails.

In [ ]:
# DO NOT do this. The model words the answer differently every run.
reply = "It is currently 31C in Mumbai with light rain."

# any of these are equally correct answers:
alternatives = [
    "Mumbai is 31 degrees.",
    "Right now in Mumbai: 31C, humid.",
    "The temperature in Mumbai is thirty-one Celsius.",
]
print("would 'exact match' pass on these correct answers?")
for a in alternatives:
    print(f"  {a == reply!s:5}  {a}")

Every one of those is a correct answer. Every one fails an exact-match
assertion. A test that flakes on correct output is worse than no test, because
people learn to ignore it.

## 5. Test the invariants instead

An invariant is a promise the output must keep, no matter how it is worded. You
cannot promise the exact sentence, but you can promise the queue is real, the
priority is valid, and nothing required is missing.

In [ ]:
# a valid triage result, however it was worded
result = {"queue": "billing", "priority": "normal", "assigned_to": "billing-team"}

checks = check_invariants(result, {
    "queue is valid":   queue_is_valid(),
    "priority present": field_present("priority"),
    "priority allowed": field_in("priority", ["normal", "urgent"]),
})

for c in checks:
    print(f"  [{'pass' if c.passed else 'FAIL'}] {c.name}")
print("all passed:", all_passed(checks))

In [ ]:
# now a result the model might produce on a bad day
bad = {"queue": "the billing dept maybe", "priority": "", "assigned_to": "x"}

checks = check_invariants(bad, {
    "queue is valid":   queue_is_valid(),
    "priority present": field_present("priority"),
})
for c in checks:
    print(f"  [{'pass' if c.passed else 'FAIL'}] {c.name}")
print("all passed:", all_passed(checks), " <- correctly rejected")

This test never flakes. Run it a thousand times, with the answer worded a
thousand ways, and it passes whenever the answer is valid and fails whenever it
is not. **That is a test you can trust in CI.** It is last lecture's run log
grown into a discipline.

## Your turn

**1. Break the memory bound.** Set a session `max_turns=3`, hold a 6-turn
conversation that refers back to turn 1, and watch context get lost. Then pin the
fact instead and watch it survive. Feel *why* pinning exists.

**2. Write an invariant test.** For your Practical 2 agent, write one test that
asserts an invariant, not exact output. Run it several times and confirm it
passes on valid answers regardless of wording.

**3. List three invariants.** Name three things your capstone system must *never*
violate. These are your test suite today and your safety argument in Unit 6, the
same idea wearing two hats.

In [ ]:
# your work here
